In [1]:
import heapq
from dataclasses import dataclass
from datetime import date, datetime, timedelta
from itertools import count
import threading
import ipywidgets as widgets
from IPython.display import clear_output, display

KATEGORI_BUMBU_VALID = [
    "Bumbu Tumis",
    "Bumbu Kuah/Soto/Sup",
    "Bumbu Goreng/Panggang",
    "Bumbu Pasta/Gulai/Rendang",
    "Bumbu Racik Instan",
    "Bumbu Tabur/Marinasi",
]


@dataclass
class FoodNode:
  nama: str
  jumlah: int
  kategori: str
  kedaluwarsa: date
  satuan: str = "bungkus"

  def heuristik(self, hari_ini: date) -> int:
    return (self.kedaluwarsa - hari_ini).days

  def status(self, hari_ini: date) -> str:
    h = self.heuristik(hari_ini)
    if h < 0:
      return "KEDALUWARSA"
    if h <= 2:
      return "SEGERA GUNAKAN"
    if h <= 7:
      return "PERHATIAN"
    return "AMAN"


class FoodWasteTracker:

  def __init__(self):
    self.inventaris: list[FoodNode] = []

  def tambah(
      self,
      nama: str,
      jumlah: int,
      kategori: str,
      kedaluwarsa: str,
      satuan: str = "bungkus",
  ):
    if jumlah <= 0:
      raise ValueError("Jumlah bungkus harus lebih dari 0!")

    kategori_clean = kategori.strip().lower()
    valid_lower = [k.lower() for k in KATEGORI_BUMBU_VALID]
    if kategori_clean not in valid_lower and "bumbu" not in nama.lower():
      raise ValueError(
          "Input ditolak! Hanya bumbu masakan instan yang diizinkan."
      )

    tgl = datetime.strptime(kedaluwarsa, "%Y-%m-%d").date()

    if tgl < date.today():
      raise ValueError(
          "Bumbu tidak dapat ditambahkan karena tanggal kedaluwarsa sudah"
          " lewat!"
      )

    self.inventaris.append(
        FoodNode(nama, jumlah, kategori.title(), tgl, satuan)
    )

  def hapus(self, nama: str) -> bool:
    awal = len(self.inventaris)
    self.inventaris = [
        n for n in self.inventaris if n.nama.lower() != nama.lower()
    ]
    return len(self.inventaris) < awal

  def gbfs(self, hari_ini: date | None = None):
    hari_ini = hari_ini or date.today()
    urutan = count()
    frontier: list[tuple[int, int, FoodNode]] = []
    for node in self.inventaris:
      heapq.heappush(frontier, (node.heuristik(hari_ini), next(urutan), node))
    hasil = []
    while frontier:
      h, _, node = heapq.heappop(frontier)
      hasil.append((node, h))
    return hasil


if "tracker" not in globals():
  tracker = FoodWasteTracker()

if len(tracker.inventaris) == 0:
  data_awal = [
      ("Bumbu Racik Soto", 2, "Bumbu Kuah/Soto/Sup", 1),
      ("Bumbu Rendang Instan", 1, "Bumbu Pasta/Gulai/Rendang", 9),
      ("Bumbu Nasi Goreng", 3, "Bumbu Racik Instan", 12),
      ("Bumbu Sayur Asem", 2, "Bumbu Kuah/Soto/Sup", 13),
      ("Bumbu Gule Instan", 1, "Bumbu Pasta/Gulai/Rendang", 15),
      ("Bumbu Tom Yum", 4, "Bumbu Racik Instan", 58),
      ("Bumbu Ayam Goreng", 3, "Bumbu Goreng/Panggang", 18),
      ("Bumbu Tumis Kangkung", 2, "Bumbu Tumis", 25),
      ("Bumbu Sup Ayam", 5, "Bumbu Kuah/Soto/Sup", 33),
      ("Bumbu Tabur Balado", 2, "Bumbu Tabur/Marinasi", 44),
  ]
  for nama, jumlah, kategori, sisa_hari in data_awal:
    exp = (date.today() + timedelta(days=sisa_hari)).strftime("%Y-%m-%d")
    tracker.tambah(nama, jumlah, kategori, exp)

html_header = widgets.HTML(
    value=f"""
    <div style='background: #f0f9ff; padding: 22px 28px; border-radius: 14px; margin-bottom: 22px; color: #0f172a; border: 1px solid #bae6fd; box-shadow: 0 4px 12px rgba(2, 132, 199, 0.05); font-family: "Segoe UI", system-ui, sans-serif;'>
        <div style='display: flex; justify-content: space-between; align-items: center;'>
            <div>
                <div style='display: flex; align-items: center; gap: 8px;'>
                    <span style='background: #e0f2fe; color: #0369a1; border: 1px solid #7dd3fc; padding: 4px 12px; border-radius: 20px; font-size: 11px; font-weight: 800; text-transform: uppercase; letter-spacing: 0.8px;'>Greedy Heuristic AI Engine</span>
                    <span style='color: #cbd5e1; font-size: 12px;'>•</span>
                    <span style='color: #0369a1; font-size: 12px; font-weight: 600;'>System Active</span>
                </div>
                <h1 style='margin: 10px 0 2px 0; font-size: 26px; font-weight: 800; letter-spacing: -0.5px; color: #0f172a;'>Racik</h1>
                <p style='margin: 0; font-size: 13px; color: #475569; font-weight: 500;'>Manajemen dan optimasi pengurangan sampah bumbu dapur instan secara real-time.</p>
            </div>
            <div style='text-align: right; background: #ffffff; padding: 10px 16px; border-radius: 12px; border: 1px solid #bae6fd;'>
                <div style='font-size: 10px; color: #0369a1; text-transform: uppercase; font-weight: 700; letter-spacing: 0.5px;'>Tanggal Acuan</div>
                <div style='font-size: 14px; font-weight: 700; color: #0284c7; margin-top: 2px;'>{date.today().strftime('%d %b %Y')}</div>
            </div>
        </div>
    </div>
    """
)

layout_full = widgets.Layout(width="100%", margin="0px 0px 12px 0px")

lbl_nama = widgets.HTML(
    "<div style='font-family: sans-serif; font-size: 12px; font-weight: 700;"
    " color: #334155; margin-bottom: 5px;'>Nama Bumbu Instan</div>"
)
input_nama = widgets.Text(
    placeholder="Contoh: Bumbu Racik Soto", layout=layout_full
)

lbl_kategori = widgets.HTML(
    "<div style='font-family: sans-serif; font-size: 12px; font-weight: 700;"
    " color: #334155; margin-bottom: 5px;'>Kategori Bumbu</div>"
)
input_kategori = widgets.Dropdown(
    options=KATEGORI_BUMBU_VALID, layout=layout_full
)

lbl_jumlah = widgets.HTML(
    "<div style='font-family: sans-serif; font-size: 12px; font-weight: 700;"
    " color: #334155; margin-bottom: 5px;'>Jumlah (Bungkus)</div>"
)
input_jumlah = widgets.BoundedIntText(
    value=1, min=1, max=999, layout=layout_full
)

lbl_exp = widgets.HTML(
    "<div style='font-family: sans-serif; font-size: 12px; font-weight: 700;"
    " color: #334155; margin-bottom: 5px;'>Tanggal Kedaluwarsa</div>"
)
input_exp = widgets.DatePicker(value=date.today(), layout=layout_full)

btn_tambah = widgets.Button(
    description="Tambah Bumbu Baru",
    button_style="",
    layout=widgets.Layout(width="100%", height="42px", margin="8px 0px 0px 0px"),
)
btn_tambah.add_class("btn-premium-save")

output_msg_tambah = widgets.Output()

lbl_hapus = widgets.HTML(
    "<div style='font-family: sans-serif; font-size: 12px; font-weight: 700;"
    " color: #991b1b; margin-bottom: 5px;'>Pilih Bumbu yang Ingin Dihapus</div>"
)
input_hapus = widgets.Dropdown(
    options=[n.nama for n in tracker.inventaris], layout=layout_full
)
btn_hapus = widgets.Button(
    description="Hapus Item Terpilih",
    button_style="",
    layout=widgets.Layout(width="100%", height="40px", margin="8px 0px 0px 0px"),
)
btn_hapus.add_class("btn-premium-delete")

card_input = widgets.VBox(
    [
        widgets.HTML(
            "<div style='border-bottom: 2px solid #0284c7; padding-bottom:"
            " 8px; margin-bottom: 16px;'><h4 style='color: #0f172a; margin: 0;"
            " font-family: sans-serif; font-weight: 800; font-size: 15px;'>Form"
            " Tambah Bumbu</h4></div>"
        ),
        lbl_nama,
        input_nama,
        lbl_kategori,
        input_kategori,
        lbl_jumlah,
        input_jumlah,
        lbl_exp,
        input_exp,
        btn_tambah,
        output_msg_tambah,
    ],
    layout=widgets.Layout(
        padding="20px",
        background_color="#ffffff",
        border="1px solid #e2e8f0",
        border_radius="14px",
        margin="0px 0px 18px 0px",
        box_shadow="0 4px 12px rgba(0,0,0,0.03)",
    ),
)

card_delete = widgets.VBox(
    [
        widgets.HTML(
            "<div style='border-bottom: 2px solid #ef4444; padding-bottom:"
            " 8px; margin-bottom: 16px;'><h4 style='color: #991b1b; margin: 0;"
            " font-family: sans-serif; font-weight: 800; font-size: 15px;'>Kelola"
            " / Hapus List</h4></div>"
        ),
        lbl_hapus,
        input_hapus,
        btn_hapus,
    ],
    layout=widgets.Layout(
        padding="20px",
        background_color="#ffffff",
        border="1px solid #fee2e2",
        border_radius="14px",
        box_shadow="0 4px 12px rgba(0,0,0,0.03)",
    ),
)

left_panel = widgets.VBox(
    [card_input, card_delete], layout=widgets.Layout(width="35%")
)

output_stats = widgets.Output()
output_table = widgets.Output()

dashboard_main = widgets.VBox(
    [
        output_stats,
        widgets.HTML(
            "<h4 style='color: #0f172a; margin: 12px 0 14px 0; font-weight: 800;"
            " font-size: 16px; font-family: sans-serif;'>Urutan Konsumsi"
            " Prioritas Teratas</h4>"
        ),
        output_table,
    ],
    layout=widgets.Layout(
        width="63%",
        padding="20px",
        border="1px solid #e2e8f0",
        border_radius="14px",
        background_color="#ffffff",
        box_shadow="0 4px 12px rgba(0,0,0,0.03)",
    ),
)

custom_style = widgets.HTML(
    value="""
    <style>
        .btn-premium-save {
            background: linear-gradient(135deg, #0284c7 0%, #0369a1 100%) !important;
            color: white !important;
            font-weight: 700 !important;
            border-radius: 8px !important;
            border: none !important;
            box-shadow: 0 4px 10px rgba(2, 132, 199, 0.2) !important;
            transition: all 0.2s ease-in-out !important;
        }
        .btn-premium-save:hover {
            background: linear-gradient(135deg, #0369a1 0%, #075985 100%) !important;
            box-shadow: 0 6px 15px rgba(2, 132, 199, 0.35) !important;
            transform: translateY(-1px);
        }
        .btn-premium-delete {
            background: linear-gradient(135deg, #ef4444 0%, #dc2626 100%) !important;
            color: white !important;
            font-weight: 700 !important;
            border-radius: 8px !important;
            border: none !important;
            box-shadow: 0 4px 10px rgba(239, 68, 68, 0.2) !important;
            transition: all 0.2s ease !important;
        }
        .btn-premium-delete:hover {
            background: linear-gradient(135deg, #dc2626 0%, #b91c1c 100%) !important;
            box-shadow: 0 6px 15px rgba(239, 68, 68, 0.35) !important;
            transform: translateY(-1px);
        }
        .widget-text, .widget-dropdown, .widget-date-picker, .widget-bounded-int-text {
            width: 100% !important;
            max-width: 100% !important;
            box-sizing: border-box !important;
        }
        .widget-text input, .widget-dropdown select, .widget-date-picker input, .widget-bounded-int-text input {
            border-radius: 8px !important;
            border: 1px solid #cbd5e1 !important;
            padding: 8px 12px !important;
            font-size: 13px !important;
            transition: all 0.2s ease !important;
            width: 100% !important;
            max-width: 100% !important;
            box-sizing: border-box !important;
            height: 38px !important;
        }
        .widget-text input:focus, .widget-dropdown select:focus, .widget-date-picker input:focus, .widget-bounded-int-text input:focus {
            border-color: #0284c7 !important;
            box-shadow: 0 0 0 3px rgba(2, 132, 199, 0.15) !important;
            outline: none !important;
        }
    </style>
    """
)


def clear_msg_tambah():
  """Secara langsung mengosongkan area widget output pesan bawaan ipywidgets."""
  output_msg_tambah.clear_output()


def reset_form_tambah():
  """Fungsi helper untuk mengosongkan/membersihkan isi form penambahan."""
  input_nama.value = ""
  input_jumlah.value = 1
  input_kategori.value = KATEGORI_BUMBU_VALID[0]
  input_exp.value = date.today()


def render_tampilan():
  input_hapus.options = [n.nama for n in tracker.inventaris]
  hari_ini = date.today()
  hasil = tracker.gbfs(hari_ini)

  total_items = sum(node.jumlah for node, _ in hasil)
  expired_count = sum(1 for node, h in hasil if h < 0)
  urgent_count = sum(1 for node, h in hasil if 0 <= h <= 2)

  with output_stats:
    clear_output(wait=True)

    stats_html = f"""
        <div style='display: flex; gap: 14px; margin-bottom: 18px; font-family: "Segoe UI", Roboto, sans-serif;'>
            <div style='flex: 1; background: #f0f9ff; border: 1px solid #bae6fd; padding: 16px; border-radius: 12px; box-shadow: 0 2px 5px rgba(0,0,0,0.01);'>
                <span style='font-size: 11px; color: #0369a1; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;'>Total Persediaan</span>
                <h3 style='margin: 6px 0 0 0; font-size: 24px; color: #0f172a; font-weight: 800;'>{total_items} <span style='font-size: 12px; font-weight: 500; color: #64748b;'>bks</span></h3>
            </div>
            <div style='flex: 1; background: #fffbeb; border: 1px solid #fde68a; padding: 16px; border-radius: 12px; box-shadow: 0 2px 5px rgba(0,0,0,0.01);'>
                <span style='font-size: 11px; color: #b45309; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;'>Sangat Urgen (&le; 2 Hari)</span>
                <h3 style='margin: 6px 0 0 0; font-size: 24px; color: #d97706; font-weight: 800;'>{urgent_count} <span style='font-size: 12px; font-weight: 500; color: #64748b;'>item</span></h3>
            </div>
            <div style='flex: 1; background: #fef2f2; border: 1px solid #fecdd3; padding: 16px; border-radius: 12px; box-shadow: 0 2px 5px rgba(0,0,0,0.01);'>
                <span style='font-size: 11px; color: #b91c1c; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;'>Kedaluwarsa</span>
                <h3 style='margin: 6px 0 0 0; font-size: 24px; color: #dc2626; font-weight: 800;'>{expired_count} <span style='font-size: 12px; font-weight: 500; color: #64748b;'>item</span></h3>
            </div>
        </div>
        """

    recommend_boxes = []
    expired_boxes = []

    for node, h in hasil:
      if h < 0:
        badge_html = (
            "<span style='background-color: #fee2e2; color: #991b1b; padding:"
            " 3px 10px; border-radius: 6px; font-size: 11px; font-weight: 700;"
            " text-transform: uppercase; display: inline-block;'>KEDALUWARSA</span>"
        )
        expired_boxes.append(
            f"{badge_html} <span style='margin-left: 8px;'><b>{node.nama}</b>"
            f" telah kedaluwarsa sejak {-h} hari lalu! Harap segera"
            " dibuang.</span>"
        )
      elif h <= 7:
        status_label = node.status(hari_ini)
        if status_label == "SEGERA GUNAKAN":
          badge_bg, badge_fg = "#fef3c7", "#92400e"
        else:
          badge_bg, badge_fg = "#fef08a", "#713f12"

        badge_html = (
            f"<span style='background-color: {badge_bg}; color: {badge_fg};"
            " padding: 3px 10px; border-radius: 6px; font-size: 11px;"
            " font-weight: 700; text-transform: uppercase; display:"
            " inline-block;'>"
            f"{status_label}</span>"
        )
        sisa_txt = f"Sisa waktu: {h} hari" if h > 0 else "Hari ini batas akhir!"
        recommend_boxes.append(
            f"{badge_html} <span style='margin-left: 8px;'>Gunakan bumbu"
            f" <b>{node.nama}</b> terlebih dahulu ({sisa_txt}).</span>"
        )

    if expired_boxes:
      stats_html += (
          "<div style='background: #fff5f5; border-left: 4px solid #ef4444;"
          " border-top: 1px solid #fee2e2; border-right: 1px solid #fee2e2;"
          " border-bottom: 1px solid #fee2e2; padding: 14px 16px; border-radius:"
          " 8px; margin-bottom: 12px; font-family: sans-serif; font-size: 13px;"
          " color: #991b1b;'>"
      )
      stats_html += (
          "<h5 style='margin: 0 0 10px 0; font-weight: 800; color: #991b1b;"
          " text-transform: uppercase; font-size: 11px; letter-spacing:"
          " 0.5px;'>PERINGATAN ITEM KEDALUWARSA (JANGAN DIKONSUMSI):</h5>"
      )
      stats_html += (
          "<ul style='margin: 0; padding-left: 0; list-style: none; line-height:"
          " 2.2;'>"
          + "".join(
              f"<li style='margin-bottom: 6px; display: flex; align-items:"
              f" center;'>{item}</li>"
              for item in expired_boxes
          )
          + "</ul>"
      )
      stats_html += "</div>"

    if recommend_boxes:
      stats_html += (
          "<div style='background: #f8fafc; border-left: 4px solid #0284c7;"
          " border-top: 1px solid #e2e8f0; border-right: 1px solid #e2e8f0;"
          " border-bottom: 1px solid #e2e8f0; padding: 14px 16px; border-radius:"
          " 8px; margin-bottom: 12px; font-family: sans-serif; font-size: 13px;"
          " color: #334155;'>"
      )
      stats_html += (
          "<h5 style='margin: 0 0 10px 0; font-weight: 800; color: #0f172a;"
          " text-transform: uppercase; font-size: 11px; letter-spacing:"
          " 0.5px;'>Rekomendasi Prioritas Masak Hari Ini:</h5>"
      )
      stats_html += (
          "<ul style='margin: 0; padding-left: 0; list-style: none; line-height:"
          " 2.2;'>"
          + "".join(
              f"<li style='margin-bottom: 6px; display: flex; align-items:"
              f" center;'>{alert}</li>"
              for alert in recommend_boxes[:3]
          )
          + "</ul>"
      )
      stats_html += "</div>"
    elif not expired_boxes:
      stats_html += (
          "<div style='background: #f0fdf4; border: 1px solid #bbf7d0;"
          " padding: 14px 16px; border-radius: 8px; margin-bottom: 12px;"
          " font-family: sans-serif; font-size: 13px; color: #166534; font-weight:"
          " 600;'>Semua bumbu masakan terpantau aman dan prima.</div>"
      )

    display(widgets.HTML(stats_html))

  with output_table:
    clear_output(wait=True)

    html_table = """
        <div style='overflow-x: auto; border-radius: 10px; border: 1px solid #e2e8f0;'>
        <table style='width: 100%; border-collapse: collapse; font-family: "Segoe UI", Roboto, sans-serif; font-size: 13px; text-align: left;'>
            <thead>
                <tr style='background-color: #f8fafc; color: #475569; border-bottom: 2px solid #cbd5e1;'>
                    <th style='padding: 12px 16px; font-weight: 700;'>No</th>
                    <th style='padding: 12px 16px; font-weight: 700;'>Nama Bahan</th>
                    <th style='padding: 12px 16px; font-weight: 700;'>Jumlah</th>
                    <th style='padding: 12px 16px; font-weight: 700;'>Kategori</th>
                    <th style='padding: 12px 16px; font-weight: 700; text-align: center;'>Sisa Hari</th>
                    <th style='padding: 12px 16px; font-weight: 700; text-align: center;'>Status</th>
                </tr>
            </thead>
            <tbody>
        """

    for i, (node, h) in enumerate(hasil, 1):
      status_label = node.status(hari_ini)

      if status_label == "KEDALUWARSA":
        badge_bg, badge_fg, row_bg = "#fee2e2", "#991b1b", "#fff5f5"
      elif status_label == "SEGERA GUNAKAN":
        badge_bg, badge_fg, row_bg = "#fef3c7", "#92400e", "#fffbeb"
      elif status_label == "PERHATIAN":
        badge_bg, badge_fg, row_bg = "#fef08a", "#713f12", "#fefce8"
      else:
        badge_bg, badge_fg, row_bg = "#dcfce7", "#15803d", "#ffffff"

      sisa_str = f"{h} hari" if h >= 0 else f"Lewat {-h} hari"

      html_table += f"""
            <tr style='background-color: {row_bg}; border-bottom: 1px solid #f1f5f9;'>
                <td style='padding: 12px 16px; font-weight: 700; color: #64748b;'>{i}</td>
                <td style='padding: 12px 16px; font-weight: 600; color: #0f172a;'>{node.nama}</td>
                <td style='padding: 12px 16px; color: #334155; font-weight: 500;'>{node.jumlah} {node.satuan}</td>
                <td style='padding: 12px 16px; color: #64748b;'>{node.kategori}</td>
                <td style='padding: 12px 16px; text-align: center; font-weight: 700; color: {badge_fg};'>{sisa_str}</td>
                <td style='padding: 12px 16px; text-align: center;'>
                    <span style='background-color: {badge_bg}; color: {badge_fg}; padding: 4px 10px; border-radius: 6px; font-size: 11px; font-weight: 700; text-transform: uppercase;'>{status_label}</span>
                </td>
            </tr>
            """

    html_table += "</tbody></table></div>"
    display(widgets.HTML(html_table))


def on_tambah_clicked(b):
  with output_msg_tambah:
    clear_output(wait=True)

    if not input_nama.value:
      return

    if input_exp.value < date.today():
      display(
          widgets.HTML(
              "<div style='background: #fef2f2; border: 1px solid #fca5a5;"
              " padding: 10px 12px; border-radius: 8px; color: #991b1b;"
              " font-weight: 600; font-size: 12px; margin-top: 10px;"
              " font-family: sans-serif; line-height: 1.4;'>⚠️ Input Ditolak:"
              " Bumbu tidak dapat ditambahkan karena tanggal kedaluwarsa sudah"
              " lewat!</div>"
          )
      )
      reset_form_tambah()
      threading.Timer(5.0, clear_msg_tambah).start()
      return

    try:
      tracker.tambah(
          input_nama.value,
          int(input_jumlah.value),
          input_kategori.value,
          input_exp.value.strftime("%Y-%m-%d"),
      )
      render_tampilan()
      reset_form_tambah()
    except Exception as e:
      display(
          widgets.HTML(
              "<div style='background: #fef2f2; border: 1px solid #fca5a5;"
              " padding: 10px 12px; border-radius: 8px; color: #991b1b;"
              " font-weight: 600; font-size: 12px; margin-top: 10px;"
              f" font-family: sans-serif;'>Gagal menambahkan bumbu: {e}</div>"
          )
      )
      reset_form_tambah()
      threading.Timer(5.0, clear_msg_tambah).start()


def on_hapus_clicked(b):
  if input_hapus.value:
    tracker.hapus(input_hapus.value)
    clear_msg_tambah()
    render_tampilan()


btn_tambah.on_click(on_tambah_clicked)
btn_hapus.on_click(on_hapus_clicked)

ui_container = widgets.HBox(
    [left_panel, dashboard_main],
    layout=widgets.Layout(justify_content="space-between", gap="20px"),
)
display(custom_style, html_header, ui_container)

render_tampilan()

HTML(value='\n    <style>\n        .btn-premium-save {\n            background: linear-gradient(135deg, #0284c…

HTML(value='\n    <div style=\'background: #f0f9ff; padding: 22px 28px; border-radius: 14px; margin-bottom: 22…